# Real-World Data Project: Healthcare Disease Risk Classification

This notebook presents a complete beginner-friendly but professional data science project using a real healthcare dataset. The workflow covers data loading, preprocessing, exploratory data analysis, visualization, feature engineering, machine learning, evaluation, and model saving.

**Domain:** Healthcare  
**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset  
**Objective:** Predict whether a tumor is benign or malignant based on clinical measurements.

## 1. Import Libraries

The project uses the required Python data science stack: pandas, numpy, matplotlib, seaborn, and scikit-learn. Joblib is used to save the trained model.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "dataset"
IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
MPL_DIR = PROJECT_DIR / ".matplotlib"

for directory in [DATASET_DIR, IMAGES_DIR, MODELS_DIR, MPL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPL_DIR.resolve())

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120

## 2. Load and Save the Dataset

The dataset is loaded from scikit-learn and saved as a CSV file in the `dataset/` folder so the project has a clear, GitHub-friendly data source.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

df["diagnosis_label"] = df["target"].map({0: "malignant", 1: "benign"})
df = df.drop(columns=["target"])

dataset_path = DATASET_DIR / "breast_cancer_wisconsin.csv"
df.to_csv(dataset_path, index=False)

print(f"Dataset saved to: {dataset_path}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

## 3. Basic Data Understanding

This section reviews data types, missing values, duplicate records, and summary statistics.

In [ ]:
print("Dataset information:")
display(df.info())

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(10))

print(f"\nDuplicate rows: {df.duplicated().sum()}")

display(df.describe().T.head(10))

## 4. Data Preprocessing

Preprocessing includes removing duplicates, handling missing values, detecting and treating outliers, creating new features, encoding the target label, and preparing the feature matrix.

In [ ]:
df_clean = df.copy()

# Remove duplicate records if present.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Fill missing numeric values with median values. This dataset has no missing values,
# but the step is included to keep the workflow production-oriented.
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill categorical missing values with mode.
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"Shape after cleaning: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isna().sum().sum()}")

In [ ]:
def cap_outliers_iqr(dataframe, columns):
    capped = dataframe.copy()
    outlier_summary = {}

    for col in columns:
        q1 = capped[col].quantile(0.25)
        q3 = capped[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = ((capped[col] < lower) | (capped[col] > upper)).sum()
        outlier_summary[col] = int(outlier_count)
        capped[col] = capped[col].clip(lower=lower, upper=upper)

    return capped, outlier_summary

feature_cols = [col for col in df_clean.columns if col != "diagnosis_label"]
df_processed, outlier_summary = cap_outliers_iqr(df_clean, feature_cols)

outlier_summary_df = (
    pd.DataFrame.from_dict(outlier_summary, orient="index", columns=["outlier_count"])
    .sort_values("outlier_count", ascending=False)
)

display(outlier_summary_df.head(10))

In [ ]:
# Feature engineering: create clinically interpretable ratio and interaction features.
df_processed["area_perimeter_ratio"] = df_processed["mean area"] / (df_processed["mean perimeter"] + 1e-6)
df_processed["compactness_ratio"] = df_processed["mean compactness"] / (df_processed["mean concavity"] + 1e-6)
df_processed["mean_texture_smoothness"] = df_processed["mean texture"] * df_processed["mean smoothness"]
df_processed["worst_area_log"] = np.log1p(df_processed["worst area"])

# Encode the categorical target variable for classification.
label_encoder = LabelEncoder()
df_processed["diagnosis_encoded"] = label_encoder.fit_transform(df_processed["diagnosis_label"])

print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
df_processed.head()

## 5. Exploratory Data Analysis

This section explores class balance, feature distributions, correlations, category-wise differences, trends, and pairwise relationships.

In [ ]:
def save_plot(filename):
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / filename, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(data=df_processed, x="diagnosis_label", hue="diagnosis_label", legend=False)
plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Patient Count")
save_plot("target_distribution.png")

In [ ]:
corr_cols = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean compactness",
    "mean concavity",
    "mean symmetry",
    "worst radius",
    "worst area",
    "diagnosis_encoded",
]

plt.figure(figsize=(10, 8))
sns.heatmap(df_processed[corr_cols].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap for Key Diagnostic Features")
save_plot("correlation_heatmap.png")

In [ ]:
plt.figure(figsize=(12, 7))
for idx, col in enumerate(["mean radius", "mean texture", "mean area", "mean smoothness"], start=1):
    plt.subplot(2, 2, idx)
    sns.histplot(data=df_processed, x=col, hue="diagnosis_label", kde=True, bins=30)
    plt.title(f"Distribution of {col.title()}")

save_plot("feature_distributions.png")

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_processed, x="diagnosis_label", y="worst area", hue="diagnosis_label", legend=False)
plt.title("Worst Area by Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Worst Area")
save_plot("boxplot_worst_area.png")

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_processed,
    x="mean radius",
    y="mean texture",
    hue="diagnosis_label",
    alpha=0.8,
)
plt.title("Mean Radius vs Mean Texture")
save_plot("scatter_radius_texture.png")

In [ ]:
pairplot = sns.pairplot(
    df_processed[
        [
            "mean radius",
            "mean texture",
            "mean smoothness",
            "worst area",
            "diagnosis_label",
        ]
    ],
    hue="diagnosis_label",
    corner=True,
    plot_kws={"alpha": 0.75, "s": 28},
)
pairplot.fig.suptitle("Pairplot of Selected Clinical Measurements", y=1.02)
pairplot.savefig(IMAGES_DIR / "pairplot_selected_features.png", bbox_inches="tight")
plt.show()

In [ ]:
category_means = (
    df_processed.groupby("diagnosis_label")[["mean radius", "mean texture", "mean area", "worst area"]]
    .mean()
    .T
)

category_means.plot(kind="bar", figsize=(10, 5))
plt.title("Average Feature Values by Diagnosis")
plt.xlabel("Feature")
plt.ylabel("Average Value")
plt.xticks(rotation=30, ha="right")
save_plot("category_wise_feature_means.png")

In [ ]:
ordered = df_processed.sort_values("mean radius").reset_index(drop=True)

plt.figure(figsize=(9, 5))
sns.lineplot(data=ordered, x=ordered.index, y="mean radius", label="Mean Radius")
sns.lineplot(data=ordered, x=ordered.index, y="worst radius", label="Worst Radius")
plt.title("Trend of Radius Measurements Across Ordered Samples")
plt.xlabel("Samples Ordered by Mean Radius")
plt.ylabel("Radius")
save_plot("trend_radius_measurements.png")

## 6. Model Training

The data is split into training and testing sets. Logistic Regression and Random Forest models are trained using scikit-learn pipelines. Scaling is included inside each pipeline so the model workflow is reproducible.

In [ ]:
model_features = [
    col for col in df_processed.columns
    if col not in ["diagnosis_label", "diagnosis_encoded"]
]

X = df_processed[model_features]
y = df_processed["diagnosis_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")),
        ]
    ),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model
    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "mse": mean_squared_error(y_test, y_pred),
        }
    )

metrics_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
display(metrics_df)

## 7. Model Evaluation

The best model is selected using F1-score, then evaluated with a classification report, confusion matrix, ROC curve, and model comparison chart.

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Best model: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
plt.figure(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=label_encoder.classes_,
    cmap="Blues",
    values_format="d",
)
plt.title(f"Confusion Matrix - {best_model_name}")
save_plot("confusion_matrix.png")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_value = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"{best_model_name} AUC = {auc_value:.3f}", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
save_plot("roc_curve.png")

In [ ]:
plot_metrics = metrics_df.melt(
    id_vars="model",
    value_vars=["accuracy", "precision", "recall", "f1_score", "roc_auc"],
)

plt.figure(figsize=(9, 5))
sns.barplot(data=plot_metrics, x="variable", y="value", hue="model")
plt.ylim(0.85, 1.01)
plt.title("Model Performance Comparison")
plt.xlabel("Metric")
plt.ylabel("Score")
save_plot("model_comparison.png")

## 8. Save the Trained Model

The best model is saved as a pickle-compatible joblib file in the `models/` folder. The saved object also includes feature names, label classes, and model metrics.

In [ ]:
model_artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "features": model_features,
    "label_encoder_classes": list(label_encoder.classes_),
    "metrics": metrics_df.to_dict(orient="records"),
}

model_path = MODELS_DIR / "breast_cancer_classifier.pkl"
joblib.dump(model_artifact, model_path)

print(f"Model saved to: {model_path}")

## 9. Final Insights and Conclusion

### Major Findings

- Malignant tumors generally show higher values for size-related measurements such as radius, perimeter, area, and worst area.
- Correlation analysis shows that several geometric features are strongly related, especially radius, perimeter, and area.
- Worst-case measurements provide strong separation between benign and malignant diagnosis groups.

### Trends Identified

- As mean radius increases, worst radius usually increases as well.
- Higher area and concavity-related values are more common in malignant observations.
- Feature distributions show visible separation between diagnosis categories for multiple predictors.

### Model Performance

- Logistic Regression and Random Forest were trained and compared.
- The best model was selected using F1-score, which is useful when both false positives and false negatives matter.
- Accuracy, precision, recall, F1-score, ROC-AUC, confusion matrix, ROC curve, and MSE were calculated.

### Healthcare Impact

This type of predictive model can help prioritize high-risk diagnostic records for review. It should support clinicians, not replace professional medical judgment.

### Limitations and Future Scope

- The dataset is small and does not include full patient histories.
- More robust validation would require external datasets.
- Future work could add cross-validation, hyperparameter tuning, model explainability, and a Streamlit dashboard.